In [ ]:
import numpy as np
import matplotlib.pyplot as plt

This is a 10 arm bandit problem. The 10 arm bandits set their mean values using a normal distribution of mean 0 and variance 1. For each action, the bandits give a reward with a mean equal to the true action value that was sampled from a normal distribution and a variance of 1. Possible actions are 0 to 9.

You can use this class to create multiple bandits.

Experiment Result:

In my experiment I tried to maximize rewards for multi armed bandit problem with n=10 , each experiment was performed for 1000 time steps for 100 trials. The MAB was solved using 7 algorithms and their results were compared which are shown below in the form of graphs. Greedy approach found the max reward early on but the rewards remained nearly constant after some time steps. In epsilon-greedy (epsilon=0.2) approach fixed-epsilon initiated exploration first but kept increasing little with time to give a better maximum reward than greedy method. Linear decay of epsilon in epsilon greedy method explored first and tried to balance exploration and exploitation with time, yeilding better rewards then fixed one . On the Other hand , epsilon greedy approach with exponential decay (alpha was found by formula using epsilon_min=0.01 and epsilon0=0.2) yeilded best rewards compared to all prior strategies and rewards kept increasing with time. Optimistic initialization of greedy algorithm with Q0=5 explored at first and reached the maximum rewards early on but kept linearly decreasing with time after that. UCB gave good results balancing exploration and exploitation and tried to maximize the rewards over time. Thompson Sampling gave good results with rewards increasing constantly with time. Epsilon-greedy with exponential decay and Thompson Sampling gave the best results so far. Exponential decay was outperforming all algorithms with constantly increasing rewards.

In [ ]:
class Bandit:
    def __init__(self):
        self.mean_values = [np.random.normal(loc=0, scale=1) for _ in range(10)]

    def reward(self, action):
        return np.random.normal(loc=self.mean_values[action], scale=1)

In [ ]:
def greedy(T=1000,n=10,bandit=None):
    Q=np.zeros(n)
    N=np.zeros(n)

    reward=[]
    for t in range(T):

        #take action giving max estimated reward
        A=np.argmax(Q)

        #reward for choosing that action
        R=bandit.reward(A)

        reward.append(R)

        N[A]+=1 #update number of times an action has been selected
        Q[A]+=(R-Q[A])/N[A] #update estimated reward for each action selected

    return reward
    

In [ ]:
def epsilon_greedy_fixed(T=1000,n=10,epsilon=0.2,bandit=None):
    Q=np.zeros(n)
    N=np.zeros(n)
    reward=[]

    for t in range(T):

        random=np.random.rand()

        #explore with epsilon and greedy with 1-epsilon
        if random>=epsilon:
            #get greedy action
            A=np.argmax(Q)
        else:
            A=np.random.randint(n)

        
        #reward for choosing that action
        R=bandit.reward(A)

        reward.append(R)

        N[A]+=1 #update number of times an action has been selected
        Q[A]+=(R-Q[A])/N[A] #update estimated reward for each action selected

    return reward


In [ ]:
def epsilon_greedy_linear(T=1000,n=10,bandit=None):
    Q=np.zeros(n)
    N=np.zeros(n)
    _epsilon=0.2
    epsilon=0
    reward=[]

    for t in range(T):

        #find k for linear decay
        k=(_epsilon-epsilon)/T
        epsilon=max(0,_epsilon-k*t)

        random=np.random.rand()

        #explore with epsilon and greedy with 1-epsilon
        if random>=epsilon:
            #get greedy action
            A=np.argmax(Q)
        else:
            A=np.random.randint(n)

        
        #reward for choosing that action
        R=bandit.reward(A)

        reward.append(R)

        N[A]+=1 #update number of times an action has been selected
        Q[A]+=(R-Q[A])/N[A] #update estimated reward for each action selected

    return reward


In [ ]:
def epsilon_greedy_exp(T=1000,n=10,bandit=None):
    Q=np.zeros(n)
    N=np.zeros(n)
    _epsilon=0.2
    epsilon_min=0.01
    reward=[]
    #find alpha for exponential decay
    alpha=np.exp(np.log(epsilon_min/_epsilon)/T)

    for t in range(T):
        
        epsilon=_epsilon*(alpha**t)
        
        random=np.random.rand()

        #explore with epsilon and greedy with 1-epsilon
        if random>=epsilon:
            #get greedy action
            A=np.argmax(Q)
        else:
            A=np.random.randint(n)

        
        #reward for choosing that action
        R=bandit.reward(A)

        reward.append(R)

        N[A]+=1 #update number of times an action has been selected
        Q[A]+=(R-Q[A])/N[A] #update estimated reward for each action selected

    return reward


In [ ]:
def optimistic_initialization(T=1000,n=10,intial_value=5,bandit=None):
    Q=np.full(n,intial_value)
    N=np.zeros(n)

    reward=[]
    for t in range(T):

        #take action giving max estimated reward
        A=np.argmax(Q)

        #reward for choosing that action
        R=bandit.reward(A)

        reward.append(R)

        N[A]+=1 #update number of times an action has been selected
        Q[A]+=(R-Q[A])/N[A] #update estimated reward for each action selected

    return reward


In [ ]:
def UCB(T=1000,n=10,c=2,bandit=None):
    Q=np.zeros(n)
    N=np.zeros(n)
    reward=[]
    for t in range(T):

        #uncertainity bonus
        if np.all(N)==0:
            u=0
        else:
            u=c*np.sqrt(np.log(t)/N)
    
        A=np.argmax(Q+u)

        #reward for choosing that action
        R=bandit.reward(A)

        reward.append(R)

        N[A]+=1 #update number of times an action has been selected
        Q[A]+=(R-Q[A])/N[A] #update estimated reward for each action selected

    return reward


In [ ]:
def Thompson_sampling(T=1000,n=10,bandit=None):
    S=np.ones(n) #to record successes for each action
    F=np.ones(n) #to record failures for each action
    
    reward=[]
    for t in range(T):

        #sampling out reward probability for each action
        r_prob=np.random.beta(S,F)
        A=np.argmax(r_prob)
        R=bandit.reward(A)

        if R>0:
            S[A]+=1
        else:
            F[A]+=1

        reward.append(R)

    return reward


In [ ]:
trials=100
Algorithms={"greedy":[],"epsilon_fixed":[],"epsilon_linear":[],"epsilon_exp":[],"optimistic":[],"UCB":[],"Thompson_sampling":[]}
Algorithm_names=["greedy","epsilon_fixed","epsilon_linear","epsilon_exp","optimistic","UCB","Thompson_sampling"]

for name in Algorithm_names:
    mean_rewards=[]
    for _ in range(trials):
        bandit=Bandit()

        if name == "greedy":
            reward=greedy(bandit=bandit)
        elif name == "epsilon_fixed":
            reward=epsilon_greedy_fixed(bandit=bandit)
        elif name == "epsilon_linear":
            reward=epsilon_greedy_linear(bandit=bandit)
        elif name == "epsilon_exp":
            reward=epsilon_greedy_exp(bandit=bandit)
        elif name == "epsilon_exp":
            reward=epsilon_greedy_exp(bandit=bandit)
        elif name == "optimistic":
            reward=optimistic_initialization(intial_value=5,bandit=bandit)
        elif name == "UCB":
            reward=UCB(bandit=bandit)
        elif name == "Thompson_sampling":
            reward=Thompson_sampling(bandit=bandit)
        
        mean_rewards.append(reward)

    mean_rewards=np.mean(mean_rewards,axis=0)
    Algorithms[name]=mean_rewards
    

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(14,10))
ax=ax.flatten()

ax[0].plot(Algorithms['greedy'],label='greedy',color="#e35656")
ax[0].set_xlabel('time steps')
ax[0].set_ylabel('Avg rewards across 100 trials')
ax[0].legend()

ax[1].plot(Algorithms['epsilon_fixed'],label='epsilon greedy fixed')
ax[1].set_xlabel('time steps')
ax[1].set_ylabel('Avg rewards across 100 trials')
ax[1].legend()

ax[2].plot(Algorithms['epsilon_linear'],label='epsilon greedy linear decay',color='green')
ax[2].set_xlabel('time steps')
ax[2].set_ylabel('Avg rewards across 100 trials')
ax[2].legend()

ax[3].plot(Algorithms['epsilon_exp'],label='epsilon greedy exp decay',color='pink')
ax[3].set_xlabel('time steps')
ax[3].set_ylabel('Avg rewards across 100 trials')
ax[3].legend()

plt.show()

In [ ]:
fig,ax=plt.subplots(3,figsize=(14,10))
ax=ax.flatten()

ax[0].plot(Algorithms['optimistic'],label='optimistic initial value',color="#e35656")
ax[0].set_xlabel('time steps')
ax[0].set_ylabel('Avg rewards across 100 trials')
ax[0].legend()

ax[1].plot(Algorithms['UCB'],label='UCB')
ax[1].set_xlabel('time steps')
ax[1].set_ylabel('Avg rewards across 100 trials')
ax[1].legend()

ax[2].plot(Algorithms['Thompson_sampling'],label='Thompson Sampling',color='green')
ax[2].set_xlabel('time steps')
ax[2].set_ylabel('Avg rewards across 100 trials')
ax[2].legend()


plt.show()